# СИМА — Полный конвейер обработки рельефа

Демонстрация и **сравнение** всех алгоритмов модуля:

| Категория | Методы |
|---|---|
| Классификация Ground | SMRF (стандартный), SMRF cut (threshold=3) |
| Фильтрация LAS | Ручная, Статистическая (μ±mσ), Перцентильная, Outlier |
| Растеризация | mean, idw, min, max |
| Сглаживание | Gaussian, Median |
| Производные | Уклоны, Экспозиции, TPI (3-масштабный), Горизонтали, Высоты |

Все растры высот — в **единой цветовой шкале**.

In [ ]:
import sys, os, shutil, json, tempfile
from pathlib import Path
import rasterio
import numpy as np
import laspy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

backend = Path('/Users/sergeyzay/Documents/НЕДРА/СИМА/sima-web/backend')
for pkg in ['packages/sima-dem-core/src', 'packages/sima-dem-ground/src',
            'packages/sima-dem-dsm/src', 'packages/sima-dem-pipeline/src']:
    sys.path.insert(0, str(backend / pkg))

from sima_dem_ground.ground import GroundProcessing, SMRFConfig, FillConfig, RasterOutputConfig
from sima_dem_dsm.dsm import DSMBuilder, DSMConfig
from sima_dem_core.curvature import CurvatureProcessing
from sima_dem_core.raster.smooth import gauss_smooth
from sima_dem_core.raster.median import med_filter
from sima_dem_core.raster.tpi import calculate_tpi, TPIConfig
from sima_dem_core.raster.contours import generate_contours
from sima_dem_core.height import get_every_nth
from sima_dem_core.filters import ManualFilter, StatFilter, RangeFilter, OutlierFilter
from sima_dem_core.check_classification import CheckClassification

print('Импорт готов')

In [ ]:
DATASET = 'test'

if DATASET == 'demo':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/pt000100.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/00000100.tif'
    REFERENCE_DSM = None
elif DATASET == 'test':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_ground_TLO.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g.tif'
    REFERENCE_DSM = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_DSM.tif'

OUTPUT_DIR = str(backend / 'output' / f'notebook_{DATASET}')
os.makedirs(OUTPUT_DIR, exist_ok=True)
RESOLUTION = 1.0

crs_source = REFERENCE_DSM if REFERENCE_DSM else TIF_PATH
with rasterio.open(crs_source) as src:
    CRS = src.crs.to_wkt()
print(f'Датасет: {DATASET}, CRS: {CRS[:60]}...')

if REFERENCE_DSM:
    sys.path.insert(0, str(backend))
    from tests.fixtures.restore_las import restore_absolute_las
    restored = str(Path(OUTPUT_DIR) / 'restored_absolute.las')
    restore_absolute_las(LAS_PATH, REFERENCE_DSM, restored)
    LAS_PATH = restored

las_orig = laspy.read(LAS_PATH)
cls_orig = np.asarray(las_orig.classification)
print(f'Точек: {len(las_orig.points):,}, классы: {sorted(set(cls_orig))}')
print(f'Z: {np.min(las_orig.z):.1f} – {np.max(las_orig.z):.1f}')

---
## 1. Сравнение методов растеризации (output_type)

In [ ]:
output_types = ['mean', 'idw', 'min', 'max']
rasterization_results = {}

for ot in output_types:
    out_dir = str(Path(OUTPUT_DIR) / f'raster_{ot}')
    os.makedirs(out_dir, exist_ok=True)
    gp = GroundProcessing(
        output=out_dir, resolution=RESOLUTION, crs=CRS,
        interpolate=False, save_ground_las=False,
        fill=FillConfig(fill_holes=False, fallback_to_min_z=False),
        raster_out=RasterOutputConfig(output_type=ot),
    )
    gp.get_raster(LAS_PATH, crs_wkt=CRS)
    rasterization_results[ot] = gp.raster[0]
    print(f'{ot}: {gp.raster[0]}')

In [ ]:
def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)
    return arr

raster_arrays = {ot: read_raster(p) for ot, p in rasterization_results.items()}

z_min = min(np.nanmin(a) for a in raster_arrays.values())
z_max = max(np.nanmax(a) for a in raster_arrays.values())
terrain_norm = mcolors.Normalize(vmin=z_min, vmax=z_max)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
for ax, (ot, arr) in zip(axes, raster_arrays.items()):
    valid = np.isfinite(arr)
    n_valid = valid.sum()
    n_total = arr.size
    pct = 100 * n_valid / n_total
    im = ax.imshow(arr, cmap='terrain', norm=terrain_norm)
    ax.set_title(f'{ot}\n{n_valid:,}/{n_total:,} ({pct:.1f}%)', fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle('Сравнение output_type (writers.gdal)', fontsize=16)
plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / 'compare_output_type.png'), dpi=150)
plt.show()

---
## 2. Сравнение методов фильтрации LAS

In [ ]:
filter_results = {'original': (len(las_orig.points), LAS_PATH)}

f_dir = str(Path(OUTPUT_DIR) / 'filters')
os.makedirs(f_dir, exist_ok=True)

# Manual
out = str(Path(f_dir) / 'manual.las')
ManualFilter(LAS_PATH, RESOLUTION, z_min=-5, z_max=30, output_file_path=out).filter()
filter_results['manual (Z: -5..30)'] = (len(laspy.read(out).points), out)

# Statistical (m=2)
out = str(Path(f_dir) / 'stat.las')
StatFilter(LAS_PATH, RESOLUTION, m=2.0, output_file_path=out).filter()
filter_results['stat (m=2σ)'] = (len(laspy.read(out).points), out)

# Percentile (0..95%)
out = str(Path(f_dir) / 'range.las')
RangeFilter(LAS_PATH, RESOLUTION, min_percent=0.0, max_percent=0.95, output_file_path=out).filter()
filter_results['percentile (0..95%)'] = (len(laspy.read(out).points), out)

# Outlier (k=8, mult=2)
out = str(Path(f_dir) / 'outlier.las')
OutlierFilter(LAS_PATH, RESOLUTION, neighbours=8, m=2.0, output_file_path=out).filter()
filter_results['outlier (k=8, m=2)'] = (len(laspy.read(out).points), out)

print(f'{"Метод":<25} {"Точек":>12} {"Удалено":>10} {"% удалено":>10}')
print('-' * 60)
orig_count = filter_results['original'][0]
for name, (count, path) in filter_results.items():
    removed = orig_count - count
    pct = 100 * removed / orig_count
    print(f'{name:<25} {count:>12,} {removed:>10,} {pct:>9.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
for ax, (name, (count, path)) in zip(axes, filter_results.items()):
    las_f = laspy.read(path)
    z = np.asarray(las_f.z)
    ax.hist(z, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(f'{name}\n{count:,} точек', fontsize=12)
    ax.set_xlabel('Z (м)')
    ax.ticklabel_format(style='plain', axis='x')
plt.suptitle('Распределение Z по методам фильтрации', fontsize=16)
plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / 'compare_filters.png'), dpi=150)
plt.show()

---
## 3. Сравнение SMRF (стандартный vs cut_smrf)

In [ ]:
smrf_results = {}

for label, cut in [('SMRF (стандарт)', False), ('SMRF cut (threshold=3)', True)]:
    out_dir = str(Path(OUTPUT_DIR) / f'smrf_{"cut" if cut else "std"}')
    os.makedirs(out_dir, exist_ok=True)
    gp = GroundProcessing(
        output=out_dir, resolution=RESOLUTION, crs=CRS,
        interpolate=True, save_ground_las=False,
        cut_smrf=cut,
        smrf=SMRFConfig(),
        fill=FillConfig(fill_holes=True, max_search_distance=100, fallback_to_min_z=True),
        raster_out=RasterOutputConfig(output_type='idw'),
    )
    gp.get_raster(LAS_PATH, crs_wkt=CRS)
    smrf_results[label] = gp.raster[0]
    print(f'{label}: {gp.raster[0]}')

In [ ]:
smrf_arrays = {label: read_raster(p) for label, p in smrf_results.items()}

s_min = min(np.nanmin(a) for a in smrf_arrays.values())
s_max = max(np.nanmax(a) for a in smrf_arrays.values())
s_norm = mcolors.Normalize(vmin=s_min, vmax=s_max)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (label, arr) in zip(axes[:2], smrf_arrays.items()):
    valid = np.isfinite(arr)
    im = ax.imshow(arr, cmap='terrain', norm=s_norm)
    ax.set_title(f'{label}\n{valid.sum():,}/{arr.size} ({100*valid.sum()/arr.size:.1f}%)', fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.7)

diff = np.abs(smrf_arrays[list(smrf_arrays.keys())[0]] - smrf_arrays[list(smrf_arrays.keys())[1]])
im = axes[2].imshow(diff, cmap='Reds')
axes[2].set_title(f'Разница\nmax ΔZ={np.nanmax(diff):.2f} м', fontsize=13)
plt.colorbar(im, ax=axes[2], shrink=0.7)
plt.suptitle('SMRF: стандартный vs cut (threshold=3)', fontsize=16)
plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / 'compare_smrf.png'), dpi=150)
plt.show()

---
## 4. Сравнение методов сглаживания (Gaussian vs Median)

In [ ]:
dtm_path = smrf_results['SMRF (стандарт)']
smooth_dir = str(Path(OUTPUT_DIR) / 'smoothing')
os.makedirs(smooth_dir, exist_ok=True)

smooth_results = {}

# Gaussian
gauss_path = str(Path(smooth_dir) / 'gauss_smooth.tif')
gauss_smooth(dtm_path, gauss_path,
             sigma=2.0 * RESOLUTION, order=0, window_size=5,
             fill_holes=True, max_search_distance=100)
smooth_results['Gaussian (σ=2)'] = gauss_path

# Median
median_path = str(Path(smooth_dir) / 'median_smooth.tif')
shutil.copy2(dtm_path, median_path)
med_filter(median_path, 5)
smooth_results['Median (window=5)'] = median_path

print('Сглаживание готово')

In [ ]:
dtm_arr = read_raster(dtm_path)
smooth_arrays = {label: read_raster(p) for label, p in smooth_results.items()}

all_z = [dtm_arr] + list(smooth_arrays.values())
z_min_s = min(np.nanmin(a) for a in all_z)
z_max_s = max(np.nanmax(a) for a in all_z)
s_norm = mcolors.Normalize(vmin=z_min_s, vmax=z_max_s)

fig, axes = plt.subplots(1, 3, figsize=(21, 6))
im = axes[0].imshow(dtm_arr, cmap='terrain', norm=s_norm)
axes[0].set_title('Исходная DTM', fontsize=13)
plt.colorbar(im, ax=axes[0], shrink=0.7)

for ax, (label, arr) in zip(axes[1:], smooth_arrays.items()):
    diff = np.abs(arr - dtm_arr)
    im = ax.imshow(arr, cmap='terrain', norm=s_norm)
    ax.set_title(f'{label}\nRMSE vs DTM={np.sqrt(np.nanmean(diff[np.isfinite(dtm_arr) & np.isfinite(arr)]**2)):.3f} м', fontsize=13)
    plt.colorbar(im, ax=ax, shrink=0.7)

plt.suptitle('Сглаживание: Gaussian vs Median', fontsize=16)
plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / 'compare_smoothing.png'), dpi=150)
plt.show()

---
## 5. Полный конвейер (все выходные продукты)

In [ ]:
dtm_path = smrf_results['SMRF (стандарт)']
smoothed_path = smooth_results['Gaussian (σ=2)']

builder = DSMBuilder(
    output=OUTPUT_DIR, crs=CRS,
    config=DSMConfig(resolution=RESOLUTION, output_type='max',
                     interpolate=True, fill_holes=True),
)
dsm_path = builder.build(LAS_PATH, crs_wkt=CRS)

cp = CurvatureProcessing()
slope_path = cp.calculate_slope(smoothed_path, CRS, RESOLUTION, RESOLUTION, OUTPUT_DIR)
aspect_path = cp.calculate_aspect(smoothed_path, CRS, RESOLUTION, RESOLUTION, OUTPUT_DIR)

tpi_path = calculate_tpi(smoothed_path, CRS, OUTPUT_DIR, RESOLUTION, 10.0,
    config=TPIConfig(radii_m=[270, 810, 2430], res=10.0))

contour_path = str(Path(OUTPUT_DIR) / 'contours.gpkg')
generate_contours(smoothed_path, interval=2.0, output_path=contour_path, crs_wkt=CRS)

ground_las_path = str(Path(OUTPUT_DIR) / 'ground_for_heights.las')
gp_h = GroundProcessing(output=OUTPUT_DIR, resolution=RESOLUTION, crs=CRS, save_ground_las=True)
gp_h.get_raster(LAS_PATH, crs_wkt=CRS, out_path=ground_las_path)
heights_path = str(Path(OUTPUT_DIR) / 'heights.geojson')
get_every_nth(ground_las_path, 10, heights_path, CRS)

print('Полный конвейер завершён')

In [ ]:
dsm = read_raster(dsm_path)
dtm = read_raster(dtm_path)
smoothed = read_raster(smoothed_path)
slope = read_raster(slope_path)
aspect = read_raster(aspect_path)
tpi = read_raster(tpi_path)

z_min = np.nanmin([np.nanmin(dtm), np.nanmin(dsm), np.nanmin(smoothed)])
z_max = np.nanmax([np.nanmax(dtm), np.nanmax(dsm), np.nanmax(smoothed)])
terrain_norm = mcolors.Normalize(vmin=z_min, vmax=z_max)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
rasters = [
    ('DTM (ЦМР)', dtm, 'terrain', terrain_norm),
    ('DSM (ЦММ)', dsm, 'terrain', terrain_norm),
    ('Сглаженная DTM', smoothed, 'terrain', terrain_norm),
    ('Уклоны (°)', slope, 'hot', None),
    ('Экспозиции (°)', aspect, 'hsv', None),
    ('TPI', tpi, 'RdBu_r', None),
]
for ax, (title, data, cmap, norm) in zip(axes.flatten(), rasters):
    im = ax.imshow(data, cmap=cmap, norm=norm)
    ax.set_title(title, fontsize=14)
    plt.colorbar(im, ax=ax, shrink=0.7)
plt.suptitle(f'СИМА — {DATASET} (resolution={RESOLUTION}m)', fontsize=16)
plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / 'overview.png'), dpi=150)
plt.show()

---
## 6. Сравнение с эталоном

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if REFERENCE_DSM:
    ref = read_raster(REFERENCE_DSM)
    min_h = min(dtm.shape[0], ref.shape[0])
    min_w = min(dtm.shape[1], ref.shape[1])
    b, r = dtm[:min_h,:min_w], ref[:min_h,:min_w]
    valid = np.isfinite(b) & np.isfinite(r)
    diff = np.where(valid, np.abs(b - r), np.nan)
    rmse = np.sqrt(np.nanmean(diff[valid]**2)) if valid.any() else 0
    mean_ref = np.nanmean(r[valid]) if valid.any() else 0
    rel_err = rmse / mean_ref if mean_ref else 0
    print(f'RMSE: {rmse:.4f} м, Mean: {mean_ref:.4f} м, Ошибка: {rel_err:.4%}')
    print(f'Требование < 5%: {"\u2713 ПРОЙДЕН" if rel_err < 0.05 else "\u2717 НЕ ПРОЙДЕН"}')

    for ax, title, data in [(axes[0], 'Построено', b), (axes[1], 'Эталон', r)]:
        im = ax.imshow(data, cmap='terrain', norm=terrain_norm)
        ax.set_title(title)
        plt.colorbar(im, ax=ax, shrink=0.7)
    im = axes[2].imshow(diff, cmap='Reds')
    axes[2].set_title(f'Разница (RMSE={rmse:.2f} м)')
    plt.colorbar(im, ax=axes[2], shrink=0.7)
else:
    print('Сравнение с эталоном недоступно (demo_data)')
    for ax in axes: ax.set_visible(False)

plt.tight_layout()
plt.show()

---
## 7. Сводная таблица метрик

In [ ]:
def raster_stats(arr, name):
    valid = arr[np.isfinite(arr)]
    return {
        'Продукт': name,
        'Размер': f'{arr.shape[1]}×{arr.shape[0]}',
        'Валидных': f'{len(valid):,} ({100*len(valid)/arr.size:.1f}%)',
        'Z min': f'{np.min(valid):.2f}' if len(valid) else '—',
        'Z max': f'{np.max(valid):.2f}' if len(valid) else '—',
        'Z mean': f'{np.mean(valid):.2f}' if len(valid) else '—',
        'Z std': f'{np.std(valid):.2f}' if len(valid) else '—',
    }

products = [
    ('DTM (IDW)', read_raster(rasterization_results['idw'])),
    ('DTM (mean)', read_raster(rasterization_results['mean'])),
    ('DTM (min)', read_raster(rasterization_results['min'])),
    ('DTM (max)', read_raster(rasterization_results['max'])),
    ('DTM SMRF std', read_raster(smrf_results['SMRF (стандарт)'])),
    ('DTM SMRF cut', read_raster(smrf_results['SMRF cut (threshold=3)'])),
    ('DTM + Gauss', read_raster(smooth_results['Gaussian (σ=2)'])),
    ('DTM + Median', read_raster(smooth_results['Median (window=5)'])),
    ('DSM (max)', dsm),
    ('Уклоны', slope),
    ('Экспозиции', aspect),
    ('TPI', tpi),
]

import pandas as pd
df = pd.DataFrame([raster_stats(arr, name) for name, arr in products])
df

In [ ]:
print(f'Выходные файлы ({OUTPUT_DIR}):')
for f in sorted(Path(OUTPUT_DIR).rglob('*')):
    if f.is_file() and f.suffix in ('.tif', '.las', '.png', '.gpkg', '.geojson'):
        size = f.stat().st_size / 1024 / 1024
        print(f'  {f.relative_to(OUTPUT_DIR)}  ({size:.1f} MB)')